# TensorFlow Recommender Demo

This notebook is a lightweight companion to the main recommender example. Its purpose is to show practical familiarity with TensorFlow / Keras on a simple MovieLens embedding model, not to build a production recommender.

## Setup

- Uses the same MovieLens 100k files as the main recommender notebook.
- Dependencies are managed through the repository `requirements.txt`.
- The model is a small matrix-factorization style network built with user and item embeddings.

## What This Demo Shows

- A compact TensorFlow / Keras workflow for a simple embedding-based recommender.
- Explicit device detection so the notebook uses GPU if available and CPU otherwise.
- Basic convergence discipline through validation monitoring and early stopping.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)
pd.set_option("display.max_colwidth", 60)

available_gpus = tf.config.list_physical_devices("GPU")
device_name = "GPU" if available_gpus else "CPU"
print(f"Running on: {device_name}")
if available_gpus:
    print(available_gpus)

I0000 00:00:1774352300.508873   36926 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Running on: CPU


W0000 00:00:1774352302.298170   36926 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Data Loading

The notebook reuses `../data/u.data` and `../data/u.item`. If the files are missing, it downloads the official MovieLens 100k archive.

In [2]:
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

required_files = ["u.data", "u.item"]
if not all((data_dir / file_name).exists() for file_name in required_files):
    zip_path = data_dir / "ml-100k.zip"
    extract_dir = data_dir / "ml-100k"
    urlretrieve(MOVIELENS_URL, zip_path)
    with ZipFile(zip_path, "r") as zip_file:
        zip_file.extractall(data_dir)
    for file_name in required_files:
        source = extract_dir / file_name
        destination = data_dir / file_name
        if source.exists() and not destination.exists():
            destination.write_bytes(source.read_bytes())

ratings = pd.read_csv(
    data_dir / "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)
movies = pd.read_csv(
    data_dir / "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["item_id", "title"],
)

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings = ratings.merge(movies, on="item_id", how="left")
ratings.head()

,user_id,item_id,rating,timestamp,title
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)


## Preprocessing And Split

For consistency with the main notebook, we use a simple time-aware split and hold out the latest rating for each eligible user.

In [3]:
user_ids = np.sort(ratings["user_id"].unique())
item_ids = np.sort(ratings["item_id"].unique())
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_index = {item_id: idx for idx, item_id in enumerate(item_ids)}

ratings["user_idx"] = ratings["user_id"].map(user_to_index)
ratings["item_idx"] = ratings["item_id"].map(item_to_index)

def make_time_split(frame, min_user_interactions=5):
    frame = frame.sort_values(["user_id", "timestamp", "item_id"]).copy()
    user_counts = frame.groupby("user_id")["item_id"].transform("size")
    eligible = frame[user_counts >= min_user_interactions]
    test_indices = eligible.groupby("user_id", sort=False).tail(1).index
    test = frame.loc[test_indices].copy().reset_index(drop=True)
    train = frame.drop(index=test_indices).reset_index(drop=True)
    return train, test

train_ratings, test_ratings = make_time_split(ratings)

validation_fraction = 0.1
validation_mask = np.random.rand(len(train_ratings)) < validation_fraction
validation_ratings = train_ratings.loc[validation_mask].reset_index(drop=True)
train_ratings = train_ratings.loc[~validation_mask].reset_index(drop=True)

print(f"Train size: {len(train_ratings):,}")
print(f"Validation size: {len(validation_ratings):,}")
print(f"Test size: {len(test_ratings):,}")

Train size: 89,120
Validation size: 9,937
Test size: 943


## Model

This Keras model learns one embedding vector per user and per movie, then predicts ratings from their dot product plus a global average.

In [4]:
n_users = len(user_ids)
n_items = len(item_ids)
embedding_dim = 32
global_mean = train_ratings["rating"].mean()

user_input = tf.keras.layers.Input(shape=(1,), name="user_idx")
item_input = tf.keras.layers.Input(shape=(1,), name="item_idx")

user_embedding = tf.keras.layers.Embedding(n_users, embedding_dim, name="user_embedding")(user_input)
item_embedding = tf.keras.layers.Embedding(n_items, embedding_dim, name="item_embedding")(item_input)

user_vector = tf.keras.layers.Flatten()(user_embedding)
item_vector = tf.keras.layers.Flatten()(item_embedding)
dot_product = tf.keras.layers.Dot(axes=1)([user_vector, item_vector])

model = tf.keras.Model(inputs=[user_input, item_input], outputs=dot_product)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss="mse")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_idx            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_idx            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 32)     │     30,176 │ user_idx[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 32)     │     53,824 │ item_idx[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 84,000 (328.12 KB)

 Trainable params: 84,000 (328.12 KB)

 Non-trainable params: 0 (0.00 B)

## Training

A few epochs are enough here because the notebook is meant to illustrate framework usage, not optimize a deep recommender.

In [5]:
centered_train_ratings = train_ratings["rating"] - global_mean
centered_validation_ratings = validation_ratings["rating"] - global_mean

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    x=[train_ratings["user_idx"], train_ratings["item_idx"]],
    y=centered_train_ratings,
    validation_data=(
        [validation_ratings["user_idx"], validation_ratings["item_idx"]],
        centered_validation_ratings,
    ),
    batch_size=1024,
    epochs=20,
    callbacks=[early_stopping],
    verbose=0,
)

pd.DataFrame(history.history)

,loss,val_loss
0,1.170049,0.963549
1,0.788373,0.856714
2,0.578954,0.899639
3,0.457612,0.966358


## Evaluation

We keep evaluation simple and use RMSE on the held-out ratings.

In [6]:
test_predictions = model.predict([test_ratings["user_idx"], test_ratings["item_idx"]], verbose=0).reshape(-1) + global_mean
rmse = float(np.sqrt(np.mean((test_predictions - test_ratings["rating"].to_numpy()) ** 2)))
print(f"Test RMSE: {rmse:.4f}")

Test RMSE: 1.0514


Interpretation: lower `RMSE` means the predicted ratings stay closer to the held-out true ratings. This is a simple signal that the model is learning useful user-item patterns, even though rating accuracy is not the same as full recommendation quality.

## Example Recommendations

Below we score all unseen movies for one user and return the top recommendations.

In [7]:
movie_lookup = movies.set_index("item_id")

def recommend_for_user(user_id, k=10):
    user_idx = user_to_index[user_id]
    all_item_indices = np.arange(n_items, dtype=np.int32).reshape(-1, 1)
    user_array = np.full(shape=(n_items, 1), fill_value=user_idx, dtype=np.int32)
    scores = model.predict([user_array, all_item_indices], verbose=0).reshape(-1) + global_mean

    seen_indices = train_ratings.loc[train_ratings["user_id"] == user_id, "item_idx"].to_numpy()
    scores[seen_indices] = -np.inf

    top_indices = np.argsort(scores)[::-1][:k]
    top_item_ids = [item_ids[idx] for idx in top_indices]
    return pd.DataFrame(
        {
            "item_id": top_item_ids,
            "title": movie_lookup.loc[top_item_ids, "title"].values,
            "predicted_score": scores[top_indices],
        }
    )

example_user_id = int(test_ratings.iloc[0]["user_id"])
recommend_for_user(example_user_id, k=10)

,item_id,title,predicted_score
0,511,Lawrence of Arabia (1962),5.109605
1,318,Schindler's List (1993),5.077187
2,513,"Third Man, The (1949)",5.025399
3,493,"Thin Man, The (1934)",5.013848
4,520,"Great Escape, The (1963)",4.989313
5,100,Fargo (1996),4.960509
6,603,Rear Window (1954),4.931568
7,285,Secrets & Lies (1996),4.890731
8,647,Ran (1985),4.886630
9,48,Hoop Dreams (1994),4.868977


## What This Demo Does Not Cover

- Richer recommender setups such as implicit-feedback training, multi-stage retrieval and ranking, or metadata-driven models.
- Broader evaluation beyond `RMSE`, such as ranking metrics, diversity, or online testing.
- Production concerns such as experiment tracking, monitoring, and deployment.